# Huawei-Inspired Healthcare Machine Learning Lab 3
## Logistic Regression for Binary Healthcare Classification

**Healthcare task:** Classify records in the Breast Cancer Wisconsin Diagnostic teaching dataset as malignant or benign.

**Dataset links:**
- Scikit-learn documentation: https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_breast_cancer.html
- Original UCI dataset page: https://archive.ics.uci.edu/dataset/17/breast+cancer+wisconsin+diagnostic

This notebook adapts the Huawei HCIA-AI V4.0 logistic regression workflow to healthcare. It keeps the same basic sequence: load data, standardise the features, train logistic regression, predict a label, and display prediction probabilities.

**Educational use only:** This model is not a clinical diagnostic system and must not be used for patient care.

## How to read this notebook

- A line beginning with `#` is a comment written for you. Python does not run it.
- Run the cells from top to bottom.
- A **feature** is a measurement used by the model.
- A **label** is the answer the model tries to predict.
- Logistic regression is used when the answer has two groups, such as group 0 and group 1.

## Learning objectives

By the end of this lab, learners should be able to:

1. Load a public healthcare dataset.
2. Understand features and binary labels.
3. Divide data into training and testing groups.
4. Standardise numerical features.
5. Train a logistic regression model.
6. Predict a class and its probability.
7. Read accuracy, sensitivity, specificity, precision, F1 score, confusion matrix, and ROC-AUC.

## Step 1 — Import the required packages

In [ ]:
# NumPy helps with numerical calculations.
import numpy as np
# Pandas helps us display data in tables, similar to a spreadsheet.
import pandas as pd
# Matplotlib helps us draw charts.
import matplotlib.pyplot as plt

# This loads the healthcare teaching dataset.
from sklearn.datasets import load_breast_cancer
# This divides the data into training and testing parts.
from sklearn.model_selection import train_test_split
# StandardScaler makes numerical features use a similar scale.
from sklearn.preprocessing import StandardScaler
# LogisticRegression is the classification model used in this lab.
from sklearn.linear_model import LogisticRegression
# A pipeline keeps the scaling and model steps together.
from sklearn.pipeline import Pipeline
# These tools calculate classification performance.
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
    roc_auc_score,
    RocCurveDisplay
)

# This fixed number makes the train/test split repeatable.
RANDOM_STATE = 42
print('Packages imported successfully.')

## Step 2 — Load the dataset

The dataset contains numerical measurements calculated from digitised images of fine-needle aspirate samples. The original dataset labels are:

- `0` = malignant
- `1` = benign

To make sensitivity and specificity easier to teach, this notebook creates a new label:

- `1` = malignant
- `0` = benign


In [ ]:
# Load the dataset as a table.
dataset = load_breast_cancer(as_frame=True)

# X contains the measurements used by the model.
X = dataset.data.copy()
# The original target uses 0 for malignant and 1 for benign.
original_target = dataset.target.copy()
# Create a new target where malignant is the positive class, labelled 1.
y = (original_target == 0).astype(int)

# Show basic information to confirm that the data loaded correctly.
print('Number of samples:', X.shape[0])
print('Number of features:', X.shape[1])
print('Class names in the original dataset:', list(dataset.target_names))
X.head()

## Step 3 — Look at the class distribution

Class distribution tells us how many examples belong to each group.

In [ ]:
# Count how many benign and malignant records are present.
class_counts = pd.Series(y).value_counts().sort_index()

# Place the counts in a clear table.
class_table = pd.DataFrame({
    'Class label': [0, 1],
    'Meaning': ['Benign', 'Malignant'],
    'Number of records': [class_counts.get(0, 0), class_counts.get(1, 0)]
})

class_table

In [ ]:
# Draw a simple bar chart of the two groups.
plt.figure(figsize=(6, 4))
plt.bar(class_table['Meaning'], class_table['Number of records'])
plt.ylabel('Number of records')
plt.title('Class Distribution')
plt.show()

## Step 4 — Select a small set of features

The full dataset contains 30 features. For a beginner-friendly lab, we use five understandable measurements:

- `mean radius`
- `mean texture`
- `mean perimeter`
- `mean area`
- `mean smoothness`

This smaller feature set helps students focus on the workflow. It is not presented as the best clinical model.

In [ ]:
# List the five measurements we want to use.
selected_features = [
    'mean radius',
    'mean texture',
    'mean perimeter',
    'mean area',
    'mean smoothness'
]

# Keep only these five columns.
X_selected = X[selected_features].copy()

# Display the first five rows.
X_selected.head()

## Step 5 — Divide the data into training and testing sets

The model learns from the training set. The test set is kept separate so we can evaluate the model on records it did not see during training.

In [ ]:
# Use 80% of the records for training and 20% for testing.
X_train, X_test, y_train, y_test = train_test_split(
    X_selected,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    # Stratify keeps a similar class balance in both groups.
    stratify=y
)

print('Training records:', len(X_train))
print('Testing records:', len(X_test))
print('Malignant proportion in training data:', round(y_train.mean(), 3))
print('Malignant proportion in testing data:', round(y_test.mean(), 3))

## Step 6 — Build the scaling and logistic regression pipeline

**In simple words:** Some measurements have much larger numbers than others. Standardisation prevents a large numerical scale from dominating the model.

In [ ]:
# Create a two-step workflow.
# Step 1: standardise the measurements.
# Step 2: train logistic regression.
model = Pipeline(steps=[
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
])

# Ask the model to learn from the training records.
model.fit(X_train, y_train)
print('Model training completed.')

## Step 7 — Predict the test records

The model produces two useful outputs:

- A predicted class: `0` or `1`
- A probability between 0 and 1

In [ ]:
# Predict the class for every test record.
y_pred = model.predict(X_test)
# Predict the probability of the positive class, which is malignant = 1.
y_probability = model.predict_proba(X_test)[:, 1]

# Display the first ten results beside the correct labels.
prediction_preview = pd.DataFrame({
    'Actual label': y_test.to_numpy()[:10],
    'Predicted label': y_pred[:10],
    'Predicted malignant probability': np.round(y_probability[:10], 3)
})

prediction_preview

## Step 8 — Calculate the confusion matrix

With malignant defined as the positive class:

- **True positive:** malignant record correctly predicted as malignant
- **True negative:** benign record correctly predicted as benign
- **False positive:** benign record incorrectly predicted as malignant
- **False negative:** malignant record incorrectly predicted as benign

In [ ]:
# Calculate the four confusion-matrix values.
cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
true_negative, false_positive, false_negative, true_positive = cm.ravel()

print('True negatives:', true_negative)
print('False positives:', false_positive)
print('False negatives:', false_negative)
print('True positives:', true_positive)

In [ ]:
# Draw the confusion matrix.
display = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=['Benign', 'Malignant']
)
display.plot(values_format='d')
plt.title('Confusion Matrix')
plt.show()

## Step 9 — Calculate common performance measures

- **Accuracy:** proportion of all predictions that were correct
- **Sensitivity:** proportion of malignant records correctly detected
- **Specificity:** proportion of benign records correctly identified
- **Precision:** among records predicted as malignant, the proportion that were malignant
- **F1 score:** balance between precision and sensitivity
- **ROC-AUC:** how well the probabilities separate the two groups across many thresholds

In [ ]:
# Calculate overall accuracy.
accuracy = accuracy_score(y_test, y_pred)
# Sensitivity is also called recall for the positive class.
sensitivity = recall_score(y_test, y_pred, pos_label=1)
# Specificity is the proportion of benign records correctly identified.
specificity = true_negative / (true_negative + false_positive)
# Precision measures how many malignant predictions were correct.
precision = precision_score(y_test, y_pred, pos_label=1)
# F1 combines precision and sensitivity.
f1 = f1_score(y_test, y_pred, pos_label=1)
# ROC-AUC uses the predicted probabilities rather than only the final class.
roc_auc = roc_auc_score(y_test, y_probability)

metrics_table = pd.DataFrame({
    'Metric': ['Accuracy', 'Sensitivity', 'Specificity', 'Precision', 'F1 score', 'ROC-AUC'],
    'Value': [accuracy, sensitivity, specificity, precision, f1, roc_auc]
})

metrics_table['Value'] = metrics_table['Value'].round(3)
metrics_table

## Step 10 — Display a detailed classification report

In [ ]:
# Show precision, recall and F1 score for each class.
print(classification_report(
    y_test,
    y_pred,
    target_names=['Benign', 'Malignant'],
    digits=3
))

## Step 11 — Draw the ROC curve

The ROC curve shows the trade-off between sensitivity and false-positive rate at different probability thresholds.

In [ ]:
# Draw the ROC curve using the model's predicted probabilities.
RocCurveDisplay.from_predictions(y_test, y_probability)
plt.title('ROC Curve')
plt.show()

## Step 12 — Predict one fictional example

The values below are fictional and are used only to demonstrate the code.

In [ ]:
# Create one fictional record using the same five columns used for training.
fictional_record = pd.DataFrame([{
    'mean radius': 16.0,
    'mean texture': 20.0,
    'mean perimeter': 105.0,
    'mean area': 850.0,
    'mean smoothness': 0.10
}])

# Predict its class and malignant probability.
fictional_class = model.predict(fictional_record)[0]
fictional_probability = model.predict_proba(fictional_record)[0, 1]

# Convert the number into a readable word.
class_name = 'Malignant' if fictional_class == 1 else 'Benign'

print('Predicted class:', class_name)
print(f'Predicted malignant probability: {fictional_probability:.3f}')

## Step 13 — Inspect the model coefficients

A positive coefficient moves the model towards the malignant class. A negative coefficient moves it towards the benign class. Coefficients describe the model's statistical pattern and must not be interpreted as proof of biological causation.

In [ ]:
# Take the trained logistic regression part out of the pipeline.
classifier = model.named_steps['classifier']

# Put each feature beside its learned coefficient.
coefficient_table = pd.DataFrame({
    'Feature': selected_features,
    'Coefficient': classifier.coef_[0]
}).sort_values('Coefficient', key=abs, ascending=False)

coefficient_table

## Optional extension — Train using all 30 features

In [ ]:
# Divide all 30 features into training and testing sets.
X_all_train, X_all_test, y_all_train, y_all_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

# Build the same scaling and logistic regression pipeline.
all_feature_model = Pipeline(steps=[
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(max_iter=2000, random_state=RANDOM_STATE))
])

# Train and evaluate the all-feature model.
all_feature_model.fit(X_all_train, y_all_train)
all_feature_predictions = all_feature_model.predict(X_all_test)
all_feature_probabilities = all_feature_model.predict_proba(X_all_test)[:, 1]

print('All-feature accuracy:', round(accuracy_score(y_all_test, all_feature_predictions), 3))
print('All-feature ROC-AUC:', round(roc_auc_score(y_all_test, all_feature_probabilities), 3))

## Student exercises

1. Use three features instead of five.
2. Add `mean concavity` and compare the results.
3. Change the test size from 20% to 30%.
4. Compare the five-feature model with the 30-feature model.
5. Count the false negatives and explain why they are important.
6. Change the probability threshold from 0.50 to 0.30 and observe how sensitivity and specificity change.
7. Explain why high test performance does not automatically prove that a model is safe for clinical deployment.

## Responsible-use notes

- This is a teaching dataset and not a replacement for pathology assessment.
- Do not enter identifiable patient data into this notebook.
- A model trained on one dataset may perform differently in another population or hospital.
- Clinical deployment requires external validation, bias assessment, governance, clinician oversight, and regulatory review.
- Probability is not certainty and must not be presented as a diagnosis.